# Baseline: приоритизация обращений

Минимальный baseline для задачи.  
Он обучает простую модель только на готовых табличных признаках из `train.csv` / `test.csv` и создает `submission.csv`.

`events.csv` можно использовать для улучшения решения, но в baseline признаки из событий не строятся.

In [1]:
from pathlib import Path

import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import average_precision_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

# Пути к данным.
ROOT = Path(".")
DATA_DIR = ROOT / "data"

TARGET = "target"

# Эти колонки не используем как признаки модели.
ID_COLUMNS = {"lead_id", "user_id"}
TIME_COLUMNS = {"assignment_ts", "assignment_date"}
NON_FEATURE_COLUMNS = ID_COLUMNS | TIME_COLUMNS | {TARGET, "split"}

RANDOM_STATE = 42

## Загрузка данных

Загружаем обучающую выборку, тестовую выборку и события.  
В baseline модель использует только `train.csv` и `test.csv`.

In [2]:
train = pd.read_csv(DATA_DIR / "train.csv")
test = pd.read_csv(DATA_DIR / "test.csv")
events = pd.read_csv(DATA_DIR / "events.csv")

print("train:", train.shape)
print("test:", test.shape)
print("events:", events.shape)
print("target mean:", train[TARGET].mean())

train: (13694, 119)
test: (4306, 118)
events: (254705, 7)
target mean: 0.20746312253541696


In [ ]:
display(train.head())
display(test.head(), )

,lead_id,user_id,assignment_ts,assignment_date,lead_source,call_center,region,car_segment,lead_channel,user_tenure_bucket,...,leadgen_prev_positive_14d,leadgen_prev_positive_30d,leadgen_prev_positive_90d,active_days_auto_1d,active_days_auto_3d,active_days_auto_7d,active_days_auto_14d,active_days_auto_30d,active_days_auto_90d,target
0,lead_f57db09ab39ae3e7,user_0000001,2026-04-22 11:56:00,2026-04-22,CRM,external,west,budget,retargeting,warm,...,0.0,0.0,0.0,0.0,0.0,0.0,4.0,9.0,26.0,0
1,lead_a6184b8a8165a27b,user_0000002,2026-04-07 14:49:00,2026-04-07,CRM,voxys,north,standard,partner,warm,...,0.0,NaN,1.0,0.0,0.0,0.0,NaN,4.0,5.0,0
2,lead_229c2a117dbac203,user_0000003,2026-04-12 17:01:00,2026-04-12,Perf,external,north,budget,retargeting,new,...,0.0,0.0,NaN,2.0,4.0,1.0,10.0,12.0,52.0,0
3,lead_16b19e58042ef905,user_0000005,2026-04-13 21:39:00,2026-04-13,Model,voxys,east,premium,partner,warm,...,0.0,1.0,0.0,0.0,1.0,0.0,3.0,2.0,NaN,1
4,lead_734c227324978a36,user_0000006,2026-04-12 18:01:00,2026-04-12,CRM,voxys,central,budget,retargeting,warm,...,0.0,0.0,0.0,0.0,0.0,0.0,1.0,2.0,9.0,0


,lead_id,user_id,assignment_ts,assignment_date,lead_source,call_center,region,car_segment,lead_channel,user_tenure_bucket,...,leadgen_prev_positive_7d,leadgen_prev_positive_14d,leadgen_prev_positive_30d,leadgen_prev_positive_90d,active_days_auto_1d,active_days_auto_3d,active_days_auto_7d,active_days_auto_14d,active_days_auto_30d,active_days_auto_90d
0,lead_97e409eb8f8c8246,user_0000000,2026-04-27 13:09:00,2026-04-27,CRM,voxys,north,commercial,retargeting,new,...,1.0,0.0,0.0,1.0,0.0,0.0,1.0,2.0,9.0,34.0
1,lead_55310edb4489f9e9,user_0000004,2026-04-24 14:48:00,2026-04-24,Model,voxys,east,standard,retargeting,warm,...,0.0,0.0,1.0,0.0,0.0,0.0,2.0,1.0,6.0,12.0
2,lead_e7f653a2c6a7eee8,user_0000019,2026-04-27 16:00:00,2026-04-27,Model,voxys,west,commercial,app,new,...,0.0,0.0,1.0,0.0,1.0,3.0,4.0,3.0,2.0,21.0
3,lead_22f8e1cfc487ac20,user_0000021,2026-04-27 11:56:00,2026-04-27,Model,external,north,commercial,web,loyal,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,4.0,3.0,11.0
4,lead_48b638b839abfac3,user_0000023,2026-04-26 10:13:00,2026-04-26,CRM,voxys,east,standard,app,warm,...,0.0,1.0,0.0,0.0,0.0,0.0,1.0,2.0,4.0,9.0


## Выбор признаков

Исключаем `lead_id`, `user_id`, timestamps и target.  
Остальные колонки используем как стартовый набор признаков.

In [3]:
# Добавляем признаки из events.csv, используя только события до момента назначения.

def add_event_features(df: pd.DataFrame, events: pd.DataFrame, id_col: str) -> pd.DataFrame:
    base = df[[id_col, "assignment_ts"]].copy()
    ev = events[[id_col, "event_ts", "event_type", "item_price_log", "src_slot", "ctx_seq"]].copy()
    ev["event_type"] = ev["event_type"].astype(str)

    merged = base.merge(ev, on=id_col, how="left")
    merged = merged[merged["event_ts"] < merged["assignment_ts"]].copy()
    merged["delta_hours"] = (merged["assignment_ts"] - merged["event_ts"]).dt.total_seconds() / 3600.0

    agg = (
        merged.groupby(id_col)
        .agg(
            total_events=("event_ts", "count"),
            item_views=("event_type", lambda s: (s == "item_view").sum()),
            searches=("event_type", lambda s: (s == "search").sum()),
            favorites=("event_type", lambda s: (s == "favorite").sum()),
            chat_opens=("event_type", lambda s: (s == "chat_open").sum()),
            call_clicks=("event_type", lambda s: (s == "call_click").sum()),
            mean_price=("item_price_log", "mean"),
            max_price=("item_price_log", "max"),
            min_price=("item_price_log", "min"),
            src_slot_nunique=("src_slot", "nunique"),
            ctx_seq_nunique=("ctx_seq", "nunique"),
            last_event_hours=("delta_hours", "min"),
        )
        .reset_index()
    )

    windows = [1, 3, 7, 14, 30]
    parts = [agg]
    for w in windows:
        mask = merged["delta_hours"] <= w * 24
        tmp = (
            merged.loc[mask]
            .groupby(id_col)
            .agg(
                **{f"{id_col}_events_{w}d": ("event_ts", "count")},
                **{f"{id_col}_item_view_{w}d": ("event_type", lambda s: (s == "item_view").sum())},
                **{f"{id_col}_search_{w}d": ("event_type", lambda s: (s == "search").sum())},
                **{f"{id_col}_favorite_{w}d": ("event_type", lambda s: (s == "favorite").sum())},
                **{f"{id_col}_chat_open_{w}d": ("event_type", lambda s: (s == "chat_open").sum())},
                **{f"{id_col}_call_click_{w}d": ("event_type", lambda s: (s == "call_click").sum())},
                **{f"{id_col}_mean_price_{w}d": ("item_price_log", "mean")},
            )
            .reset_index()
        )
        parts.append(tmp)

    out = base[[id_col]].copy()
    for feat in parts:
        out = out.merge(feat, on=id_col, how="left")

    return out.fillna(0)


# Приводим временные колонки к правильному типу.
train["assignment_ts"] = pd.to_datetime(train["assignment_ts"])
test["assignment_ts"] = pd.to_datetime(test["assignment_ts"])
events["event_ts"] = pd.to_datetime(events["event_ts"])

train_event_features = add_event_features(train, events, "lead_id")
user_event_features = add_event_features(train, events, "user_id")
test_event_features = add_event_features(test, events, "lead_id")
test_user_event_features = add_event_features(test, events, "user_id")

train = train.merge(train_event_features, on="lead_id", how="left").merge(user_event_features, on="user_id", how="left")
test = test.merge(test_event_features, on="lead_id", how="left").merge(test_user_event_features, on="user_id", how="left")

feature_columns = [
    column for column in train.columns
    if column not in NON_FEATURE_COLUMNS and column in test.columns
]

numeric_columns = [
    column for column in feature_columns
    if pd.api.types.is_numeric_dtype(train[column])
]

categorical_columns = [
    column for column in feature_columns
    if column not in numeric_columns
]

print("numeric:", len(numeric_columns))
print("categorical:", len(categorical_columns))
print("total features:", len(feature_columns))

numeric: 201
categorical: 7
total features: 208


## Валидация

Так как тестовая выборка находится позже train по времени, лучше валидироваться на последних датах train.  
Это ближе к реальному сценарию, чем случайный split.

In [18]:
def make_validation_split(train_df: pd.DataFrame) -> tuple[pd.DataFrame, pd.DataFrame]:
    """Делит train по времени: ранние даты в обучение, поздние даты в валидацию."""
    if "assignment_date" in train_df.columns:
        dates = pd.to_datetime(train_df["assignment_date"]).dt.date
        ordered_dates = sorted(dates.unique())
        cutoff = ordered_dates[int(len(ordered_dates) * 0.8)]

        train_part = train_df[dates < cutoff]
        valid_part = train_df[dates >= cutoff]
        return train_part, valid_part

    # Fallback, если даты нет.
    return train_test_split(
        train_df,
        test_size=0.2,
        random_state=RANDOM_STATE,
        stratify=train_df[TARGET],
    )


train_part, valid_part = make_validation_split(train)

print("train_part:", train_part.shape)
print("valid_part:", valid_part.shape)

train_part: (10272, 119)
valid_part: (3422, 119)


## Модель

Используем простую Logistic Regression:

- числовые признаки: заполнение пропусков медианой и scaling;
- категориальные признаки: заполнение самым частым значением и one-hot encoding;
- `class_weight="balanced"` из-за дисбаланса классов.

In [5]:
numeric_preprocessor = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
    ]
)

categorical_preprocessor = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(handle_unknown="ignore")),
    ]
)

preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_preprocessor, numeric_columns),
        ("cat", categorical_preprocessor, categorical_columns),
    ],
    remainder="drop",
)

model = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        ("model", LogisticRegression(max_iter=1000, class_weight="balanced")),
    ]
)

In [6]:
# Обучаемся на ранней части train и проверяем качество на поздней части train.
if "assignment_date" in train.columns:
    dates = pd.to_datetime(train["assignment_date"]).dt.date
    ordered_dates = sorted(dates.unique())
    cutoff = ordered_dates[int(len(ordered_dates) * 0.8)]

    train_part = train[dates < cutoff]
    valid_part = train[dates >= cutoff]
else:
    train_part, valid_part = train_test_split(
        train,
        test_size=0.2,
        random_state=RANDOM_STATE,
        stratify=train[TARGET],
    )

print("train_part:", train_part.shape)
print("valid_part:", valid_part.shape)

model.fit(train_part[feature_columns], train_part[TARGET])

valid_scores = model.predict_proba(valid_part[feature_columns])[:, 1]
valid_ap = average_precision_score(valid_part[TARGET], valid_scores)

print(f"Validation Average Precision: {valid_ap:.5f}")

train_part: (10272, 213)
valid_part: (3422, 213)
Validation Average Precision: 0.62028


## Submission

Обучаем модель на всем train и строим score для test.  
Файл для отправки должен содержать две колонки: `lead_id` и `score`.

In [7]:
# Финально обучаем модель на всей обучающей выборке.
model.fit(train[feature_columns], train[TARGET])

test_scores = model.predict_proba(test[feature_columns])[:, 1]

submission = pd.DataFrame(
    {
        "lead_id": test["lead_id"].astype(str),
        "score": test_scores,
    }
)

submission.to_csv(ROOT / "submission.csv", index=False)
submission.head()

,lead_id,score
0,lead_97e409eb8f8c8246,0.854053
1,lead_55310edb4489f9e9,0.542492
2,lead_e7f653a2c6a7eee8,0.965039
3,lead_22f8e1cfc487ac20,0.295562
4,lead_48b638b839abfac3,0.379330


In [8]:
# Минимальные проверки формата перед загрузкой.
assert list(submission.columns) == ["lead_id", "score"]
assert len(submission) == len(test)
assert submission["lead_id"].is_unique
assert submission["score"].between(0, 1).all()

print("submission.csv is ready")

submission.csv is ready
